# 01 - Data loading and validation

Reads the instrument's non-standard BMP exports, recovers acquisition
metadata from the filenames, and writes the clean image stack that every
later stage consumes.

> This notebook is a thin wrapper around the `src/` package. Every computation
> below is the same function the CLI calls, so the notebook and
> `python scripts/run_pipeline.py` produce identical results. To change a
> parameter, edit `configs/pipeline_config.yaml` rather than the code here.

In [1]:
import sys
from pathlib import Path

# Locate the project root so `src` imports work wherever Jupyter was started from.
ROOT = Path.cwd()
while not (ROOT / "configs" / "pipeline_config.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

%matplotlib inline
from src.config import load_config

config = load_config()
# Show paths relative to the project root so notebooks stay portable.
print("Project  :", ROOT.name)
print("Raw data :", config.raw_dir.relative_to(ROOT) if ROOT in config.raw_dir.parents else config.raw_dir)
print("Outputs  :", config.processed_dir.relative_to(ROOT) if ROOT in config.processed_dir.parents else config.processed_dir)

Project  : Fossil Fly
Raw data : data\raw
Outputs  : data\processed


## Parse the raw exports

The exporter writes a 54-byte BMP header claiming 64bpp, then interleaved
`uint16` planes. PIL and OpenCV both misread this, so the payload is parsed
directly from the bytes.

In [2]:
from src.preprocessing.load_images import load_images_from_config

images = load_images_from_config(config)
for key, image in images.items():
    print(f"{image.shape}  max={image.max():8.0f}  {key}")

(640, 640)  max=    8192  Fly_Neg_C60_30pA_10x10x64pix_400sh-35V_103.91326±0.02297u
(640, 640)  max=    8192  Fly_Neg_C60_30pA_10x10x64pix_400sh-35V_123.94783±0.02297u
(640, 640)  max=    8192  Fly_Neg_C60_30pA_10x10x64pix_400sh-35V_62.96198±0.01675u
(640, 640)  max=    8192  Fly_Neg_C60_30pA_10x10x64pix_400sh-35V_78.94747±0.01675u
(640, 640)  max=    8192  Fly_Neg_C60_30pA_10x10x64pix_400sh-35V_96.93916±0.02297u
(896, 640)  max=    8192  Fly_Pos_C60_30pA_10x14x64pix_100sh_53.93733±0.015u
(896, 640)  max=    8192  FossilFly Pos 8mm x 11.2mm 53.93733±0.015u


## Recover acquisition metadata

Each field is matched independently, so a filename missing one field still
yields the rest instead of failing.

In [3]:
import pandas as pd

from src.preprocessing.metadata import (
    attach_shapes,
    build_metadata_from_config,
    metadata_to_frame,
)

metadata = attach_shapes(build_metadata_from_config(list(images), config), images)
metadata_to_frame(metadata)[
    ["polarity", "mass", "shots", "current", "dimensions", "shape"]
]

,polarity,mass,shots,current,dimensions,shape
key,,,,,,
Fly_Neg_C60_30pA_10x10x64pix_400sh-35V_103.91326±0.02297u,Neg,103.91326±0.02297u,400.0,30pA,10x10x64pix,"(640, 640)"
Fly_Neg_C60_30pA_10x10x64pix_400sh-35V_123.94783±0.02297u,Neg,123.94783±0.02297u,400.0,30pA,10x10x64pix,"(640, 640)"
Fly_Neg_C60_30pA_10x10x64pix_400sh-35V_62.96198±0.01675u,Neg,62.96198±0.01675u,400.0,30pA,10x10x64pix,"(640, 640)"
Fly_Neg_C60_30pA_10x10x64pix_400sh-35V_78.94747±0.01675u,Neg,78.94747±0.01675u,400.0,30pA,10x10x64pix,"(640, 640)"
Fly_Neg_C60_30pA_10x10x64pix_400sh-35V_96.93916±0.02297u,Neg,96.93916±0.02297u,400.0,30pA,10x10x64pix,"(640, 640)"
Fly_Pos_C60_30pA_10x14x64pix_100sh_53.93733±0.015u,Pos,53.93733±0.015u,100.0,30pA,10x14x64pix,"(896, 640)"
FossilFly Pos 8mm x 11.2mm 53.93733±0.015u,Pos,53.93733±0.015u,NaN,NaN,NaN,"(896, 640)"


## Sanity checks and figures

In [4]:
from src.preprocessing.stack_io import summary_statistics

summary_statistics(images)

,image,shape,min,max,mean,median,std,zeros_pct,nonzero_pixels,all_finite,non_negative
0,Fly_Neg_C60_30pA_10x10x64pix_400sh-35V_103.913...,640x640,0.0,8192.0,26.147214,0.0,323.069120,98.212891,7320,True,True
1,Fly_Neg_C60_30pA_10x10x64pix_400sh-35V_123.947...,640x640,0.0,8192.0,192.196648,0.0,890.731735,91.561768,34563,True,True
2,Fly_Neg_C60_30pA_10x10x64pix_400sh-35V_62.9619...,640x640,0.0,8192.0,14.774089,0.0,281.068052,99.456787,2225,True,True
3,Fly_Neg_C60_30pA_10x10x64pix_400sh-35V_78.9474...,640x640,0.0,8192.0,39.202603,0.0,416.843015,97.175293,11570,True,True
4,Fly_Neg_C60_30pA_10x10x64pix_400sh-35V_96.9391...,640x640,0.0,8192.0,8.330098,0.0,183.694651,99.438965,2298,True,True
5,Fly_Pos_C60_30pA_10x14x64pix_100sh_53.93733±0....,896x640,0.0,8192.0,28.534741,0.0,348.457004,98.450056,8888,True,True
6,FossilFly Pos 8mm x 11.2mm 53.93733±0.015u,896x640,0.0,8192.0,28.534741,0.0,348.457004,98.450056,8888,True,True


In [5]:
from src.viz import overview

overview.plot_raw_overview(images, config.figure_path("01_raw_overview.png"))
overview.plot_intensity_histograms(
    images, config.figure_path("01_intensity_histograms.png")
)
print("figures written")

figures written


## Save the clean stack

Stored as a keyed archive rather than one cube, because a dataset may mix
acquisition geometries that cannot share a single array.

In [6]:
from src.preprocessing.stack_io import save_clean_stack

written = save_clean_stack(images, metadata, config.processed_dir)
for name, path in written.items():
    print(f"{name:10s} -> {path.name}")

stack      -> fossilfly_clean_stack.npz
metadata   -> metadata.json
summary    -> 01_summary_statistics.csv


Equivalent CLI command:

```bash
python scripts/run_pipeline.py --stage load
```